# Training YOLOv11-Pose per Field Skeleton Detection del Campo da Padel

Questo notebook allena un modello YOLOv11-pose per riconoscere la geometria completa del campo da padel usando un skeleton di 10 keypoint.

**Obiettivo**: Rilevare 10 keypoint fissi del campo (4 angoli + 4 punti rete + 2 linee servizio) per permettere:
- Ricostruzione geometria campo (perimetro + rete)
- Trasformazione prospettica (homography) per vista 2D dall'alto
- Divisione giocatori per squadra basata su posizione rete

**Dataset**: 
- 1 classe: `field`
- Keypoint shape: 10 keypoint per campo con 3 valori (x, y, visibility)
- Schema keypoint: 0-3=angoli campo, 4-7=pali rete, 8-9=linee servizio
- Split: train/valid/test

**Modello**: YOLOv11-pose (pre-trained su COCO-pose) - transfer learning per keypoint detection del campo

## 1. Configurazione Ambiente e Installazione Dipendenze

In [ ]:
# Verifica GPU disponibile e ambiente di esecuzione
import torch
import sys
import os

# Rileva ambiente
IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
IS_COLAB = 'google.colab' in sys.modules
IS_LOCAL = not (IS_KAGGLE or IS_COLAB)

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print()
print(f"🖥️  Ambiente: {'Kaggle' if IS_KAGGLE else ('Google Colab' if IS_COLAB else 'Local')}")
print()
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    gpu_count = torch.cuda.device_count()
    print(f"GPU count: {gpu_count}")
    for i in range(gpu_count):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")
    print()
    print("✅ Training su GPU - ottimo per velocità!")
else:
    print("⚠️ WARNING: GPU not available! Training will be slow on CPU.")
    if IS_LOCAL and sys.platform == 'darwin':
        print("💡 Su Mac: Verrà usato MPS (Metal Performance Shaders) se disponibile.")

In [ ]:
# Installa/aggiorna Ultralytics all'ultima versione per supporto YOLOv11
!pip install -U ultralytics -q

# Importa librerie necessarie
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import yaml
import shutil
from IPython.display import Image, display
import os
import ultralytics

print("✅ Ultralytics installato/aggiornato con successo!")
print(f"📦 Ultralytics version: {ultralytics.__version__}")
print("ℹ️  YOLOv11 richiede Ultralytics >= 8.3.0")

## 2. Caricamento Dataset

**Dataset su Kaggle**:
1. Carica il dataset come **Kaggle Dataset** o **Input** del notebook
2. Il dataset sarà disponibile in `/kaggle/input/[nome-dataset]/`
3. Aggiorna la variabile `DATASET_ROOT` qui sotto con il path corretto

**Dataset locale**:
Il dataset `Padel Keypoints` deve trovarsi in: `../data/Padel Keypoints/`

Struttura richiesta:
```
[DATASET_ROOT]/
├── data.yaml
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

In [ ]:
# Path al dataset - auto-detect basato su ambiente
import os
from pathlib import Path

if IS_KAGGLE:
    # Su Kaggle: cerca il dataset negli input
    kaggle_input_dir = Path("/kaggle/input")
    
    # Lista tutti i dataset disponibili
    available_datasets = [d.name for d in kaggle_input_dir.iterdir() if d.is_dir()]
    print(f"📂 Dataset disponibili su Kaggle: {available_datasets}")
    print()
    
    # Cerca dataset con nome contenente "padel" o "keypoint"
    padel_datasets = [d for d in available_datasets if 'padel' in d.lower() or 'keypoint' in d.lower()]
    
    if padel_datasets:
        DATASET_ROOT = kaggle_input_dir / padel_datasets[0]
        print(f"✅ Dataset rilevato automaticamente: {DATASET_ROOT}")
    else:
        # Fallback: usa il primo dataset o chiedi manualmente
        print("⚠️ Nessun dataset 'padel' trovato automaticamente.")
        print("💡 Specifica manualmente il nome del dataset:")
        print(f"   Disponibili: {available_datasets}")
        # Usa il primo disponibile come fallback
        if available_datasets:
            DATASET_ROOT = kaggle_input_dir / available_datasets[0]
            print(f"📦 Usando: {DATASET_ROOT}")
        else:
            raise FileNotFoundError("Nessun dataset trovato in /kaggle/input/. Aggiungi il dataset come input del notebook!")
    
    DATA_YAML = DATASET_ROOT / "data.yaml"
    print(f"🟢 Ambiente: Kaggle")
    
elif IS_COLAB:
    # Su Google Colab: mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_ROOT = Path("/content/drive/MyDrive/Padel_Keypoints")
    DATA_YAML = DATASET_ROOT / "data.yaml"
    print("🟢 Ambiente: Google Colab")
    print("💡 Assicurati che il dataset sia in: /content/drive/MyDrive/Padel_Keypoints")
    
else:
    # Locale
    DATASET_ROOT = Path("../data/Padel Keypoints")
    DATA_YAML = DATASET_ROOT / "data.yaml"
    print("🟢 Ambiente: Esecuzione Locale")

print(f"📂 Dataset path: {DATASET_ROOT.absolute()}")
print()

# Verifica che il dataset esista
if not DATASET_ROOT.exists():
    print(f"❌ ERROR: Dataset non trovato in {DATASET_ROOT}")
    if IS_KAGGLE:
        print("💡 Su Kaggle: Aggiungi il dataset come input del notebook!")
    elif IS_COLAB:
        print("💡 Su Colab: Carica il dataset su Google Drive!")
    else:
        print("💡 Locale: Assicurati di aver scaricato il dataset annotato da Roboflow!")
    raise FileNotFoundError(f"Dataset non trovato: {DATASET_ROOT}")
else:
    print(f"✅ Dataset trovato in: {DATASET_ROOT}")
    
# Verifica struttura cartelle
required_folders = ["train/images", "train/labels", "valid/images", "valid/labels", 
                   "test/images", "test/labels"]
missing = []
for folder in required_folders:
    folder_path = DATASET_ROOT / folder
    if not folder_path.exists():
        missing.append(folder)

if missing:
    print(f"⚠️ WARNING: Cartelle mancanti: {missing}")
else:
    print("✅ Struttura dataset verificata correttamente!")

## 3. Verifica e Analisi Dataset

In [ ]:
# Leggi e analizza data.yaml
with open(DATA_YAML, 'r') as f:
    data_config = yaml.safe_load(f)
    
print("📋 Configurazione Dataset:")
print(f"  - Numero classi: {data_config['nc']}")
print(f"  - Nomi classi: {data_config['names']}")
print(f"  - Keypoint shape: {data_config['kpt_shape']}")
print(f"  - Flip index: {data_config['flip_idx']}")
print()

# Conta immagini e labels per ogni split
splits = ["train", "valid", "test"]
for split in splits:
    images_path = DATASET_ROOT / split / "images"
    labels_path = DATASET_ROOT / split / "labels"
    
    n_images = len(list(images_path.glob("*.jpg"))) if images_path.exists() else 0
    n_labels = len(list(labels_path.glob("*.txt"))) if labels_path.exists() else 0
    
    print(f"{split.upper()}: {n_images} immagini, {n_labels} labels")
    
    if n_images != n_labels:
        print(f"  ⚠️ WARNING: Numero di immagini e labels non corrisponde!")

In [ ]:
# Visualizza esempi di annotazioni con 10 keypoint skeleton
def parse_yolo_keypoints(label_file):
    """Parse YOLO format keypoint annotations"""
    with open(label_file, 'r') as f:
        lines = f.readlines()
    
    keypoints = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 35:  # class_id + bbox (4) + 10 keypoints × 3 = 35 values
            continue
        
        class_id = int(parts[0])
        # YOLO format: class_id cx cy w h kpt_x kpt_y visibility ...
        cx, cy, w, h = map(float, parts[1:5])
        
        # Keypoints: x, y, visibility (repeats for each keypoint)
        kpts = []
        for i in range(5, len(parts), 3):
            if i+2 < len(parts):
                kpt_x, kpt_y, visibility = map(float, parts[i:i+3])
                kpts.append((kpt_x, kpt_y, visibility))
        
        keypoints.append({
            'class_id': class_id,
            'bbox': (cx, cy, w, h),
            'keypoints': kpts
        })
    
    return keypoints

def visualize_sample(image_path, label_path, class_names):
    """Visualizza un'immagine con field skeleton (10 keypoint)"""
    # Leggi immagine
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # Parse annotations
    annotations = parse_yolo_keypoints(label_path)
    
    # Colori per keypoint
    kpt_colors = [
        (255, 0, 0),     # 0: corner_back_left (rosso)
        (255, 127, 0),   # 1: corner_back_right (arancione)
        (0, 255, 0),     # 2: corner_front_left (verde)
        (0, 255, 127),   # 3: corner_front_right (ciano)
        (0, 0, 255),     # 4: net_top_left (blu)
        (127, 0, 255),   # 5: net_top_right (viola)
        (255, 0, 255),   # 6: net_bottom_left (magenta)
        (255, 0, 127),   # 7: net_bottom_right (rosa)
        (255, 255, 0),   # 8: service_line_top (giallo)
        (0, 255, 255),   # 9: service_line_bottom (ciano chiaro)
    ]
    
    for ann in annotations:
        class_id = ann['class_id']
        cx, cy, bw, bh = ann['bbox']
        
        # Converti coordinate normalizzate in pixel
        cx_px, cy_px = int(cx * w), int(cy * h)
        bw_px, bh_px = int(bw * w), int(bh * h)
        
        # Bounding box
        x1 = int(cx_px - bw_px/2)
        y1 = int(cy_px - bh_px/2)
        x2 = int(cx_px + bw_px/2)
        y2 = int(cy_px + bh_px/2)
        
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Label
        label = class_names[class_id]
        cv2.putText(img, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 
                   0.6, (0, 255, 0), 2)
        
        # Keypoints con colori diversi
        for idx, (kpt_x, kpt_y, visibility) in enumerate(ann['keypoints']):
            if visibility > 0:  # Solo keypoints visibili
                kpt_x_px = int(kpt_x * w)
                kpt_y_px = int(kpt_y * h)
                color = kpt_colors[idx % len(kpt_colors)]
                
                # Disegna keypoint
                cv2.circle(img, (kpt_x_px, kpt_y_px), 8, color, -1)
                cv2.circle(img, (kpt_x_px, kpt_y_px), 10, (255, 255, 255), 2)
                
                # Numero keypoint
                cv2.putText(img, str(idx), (kpt_x_px+12, kpt_y_px+5), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    
    return img

# Visualizza 4 esempi dal training set
train_images = sorted((DATASET_ROOT / "train" / "images").glob("*.jpg"))
class_names = data_config['names']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, img_path in enumerate(train_images[:4]):
    label_path = DATASET_ROOT / "train" / "labels" / f"{img_path.stem}.txt"
    
    if label_path.exists():
        img = visualize_sample(img_path, label_path, class_names)
        axes[i].imshow(img)
        axes[i].set_title(f"Training Sample {i+1}: {img_path.name}", fontsize=10)
        axes[i].axis('off')

plt.tight_layout()
plt.show()

print("✅ Esempi visualizzati correttamente!")
print("\nLegenda keypoint:")
print("  0: corner_back_left (rosso)")
print("  1: corner_back_right (arancione)")
print("  2: corner_front_left (verde)")
print("  3: corner_front_right (ciano)")
print("  4-7: net points (blu/viola/magenta/rosa)")
print("  8-9: service lines (giallo/ciano chiaro)")

## 4. Configurazione Modello YOLOv11-pose

**Scelta del modello YOLOv11** (più accurato e performante rispetto a YOLOv8):
- `yolov11n-pose.pt`: Nano - Veloce, leggero (2.9M params)
- `yolov11s-pose.pt`: Small - Bilanciato (9.4M params)
- `yolov11m-pose.pt`: Medium - Accurato ⭐ **RACCOMANDATO** (20.9M params)
- `yolov11l-pose.pt`: Large - Molto accurato (51.1M params)
- `yolov11x-pose.pt`: Extra Large - Massima accuratezza (58.8M params)

Per il riconoscimento del field skeleton (10 keypoint geometrici), utilizziamo **yolov11m-pose** (pre-trained su COCO-pose 17 keypoint) per ottenere alta accuratezza tramite transfer learning.

**Differenza chiave vs modello precedente**:
- Vecchio: 4 classi × 1 keypoint sparso ciascuna → molti keypoint non strutturati
- Nuovo: 1 classe × 10 keypoint strutturati → skeleton geometrico coerente

In [ ]:
# Aggiorna data.yaml con path assoluti per YOLO
updated_yaml_path = DATASET_ROOT / "data_absolute.yaml"

with open(DATA_YAML, 'r') as f:
    data_config = yaml.safe_load(f)

# Aggiorna i path relativi con path assoluti
data_config['train'] = str(DATASET_ROOT / "train" / "images")
data_config['val'] = str(DATASET_ROOT / "valid" / "images")
data_config['test'] = str(DATASET_ROOT / "test" / "images")

# Salva il nuovo data.yaml
with open(updated_yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"✅ data.yaml aggiornato con path assoluti salvato in: {updated_yaml_path}")
print()
print("Configurazione finale:")
print(f"  - Train: {data_config['train']}")
print(f"  - Val: {data_config['val']}")
print(f"  - Test: {data_config['test']}")
print(f"  - Classes: {data_config['nc']}")
print(f"  - Keypoint shape: {data_config['kpt_shape']}")

In [ ]:
# Verifica versione Ultralytics e scarica modello pre-trained
import ultralytics
print(f"📦 Ultralytics version: {ultralytics.__version__}")
print()

# YOLOv11 è molto recente - se non disponibile usa YOLOv8 (comunque eccellente!)
# YOLOv11 richiede Ultralytics >= 8.3.0
MODEL_NAME = None
model = None

# Lista di modelli da provare in ordine di preferenza
models_to_try = [
    ("yolo11m-pose.pt", "YOLOv11-Medium-Pose"),
    ("yolov8m-pose.pt", "YOLOv8-Medium-Pose"),
    ("yolov8s-pose.pt", "YOLOv8-Small-Pose")
]

for model_file, model_desc in models_to_try:
    try:
        print(f"📥 Tentativo caricamento {model_desc}: {model_file}")
        model = YOLO(model_file)
        MODEL_NAME = model_file
        print(f"✅ {model_desc} caricato con successo!")
        break
    except Exception as e:
        print(f"❌ {model_desc} non disponibile: {str(e)[:100]}")
        print()

if model is None:
    raise RuntimeError("Nessun modello YOLO-pose disponibile! Verifica l'installazione di Ultralytics.")

print()
print(f"🎯 Checkpoint: COCO-pose (17 keypoints umani) - verrà adattato per keypoints del campo")
print(f"  - Modello finale: {MODEL_NAME}")
print(f"  - Numero parametri: {sum(p.numel() for p in model.model.parameters()) / 1e6:.2f}M")
print(f"  - Device: {model.device}")

## 5. Training del Modello

**Parametri di training**:
- `epochs`: Numero di epoche (150-200 raccomandato)
- `imgsz`: Dimensione immagini input (640 standard)
- `batch`: Batch size (32-64 con 2 GPU Kaggle, 16 locale)
- `lr0`: Learning rate iniziale (0.01 default)
- `patience`: Early stopping patience (50 epoche senza miglioramento)
- `device`: 0 per GPU singola, [0,1] per multi-GPU, 'mps' per Apple Silicon, 'cpu' per CPU
- `project`: Directory dove salvare i risultati

**Configurazione GPU**:
- **Kaggle (2 GPU)**: Batch 32-64, device 0 (YOLO gestisce automaticamente multi-GPU)
- **Google Colab (1 GPU)**: Batch 16-32
- **Local GPU**: Batch 16-32 (dipende da VRAM)
- **Local CPU/MPS**: Batch 8-16

**Tempo stimato**:
- 2 GPU Kaggle: ~30-60 min per 150 epochs
- 1 GPU potente: ~1-2 ore
- Apple Silicon (MPS): ~2-3 ore
- CPU: Non consigliato (molte ore)

In [ ]:
# Parametri di training ottimizzati per GPU
project_name = "field_skeleton_" + MODEL_NAME.replace(".pt", "").replace("-", "_")

# Auto-detect device ottimale
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    
    # IMPORTANTE: device=0 con multi-GPU → YOLO usa automaticamente DataParallel
    # NON usare device=[0,1] che forza DDP (problematico su Kaggle)
    device = 0
    
    # Multi-GPU: aumenta batch size (YOLO distribuisce automaticamente)
    if gpu_count >= 2:
        if IS_KAGGLE:
            batch_size = 32  # YOLO distribuisce automaticamente su 2 GPU
            workers = 8
            print(f"🚀 Kaggle Multi-GPU: 2xT4 rilevate")
            print(f"   YOLO userà automaticamente entrambe le GPU (DataParallel)")
        else:
            batch_size = 40
            workers = 8
            print(f"🚀 Multi-GPU rilevate: {gpu_count} GPU")
            print(f"   YOLO userà automaticamente tutte le GPU")
    else:
        device = 0  # Singola GPU
        if IS_KAGGLE:
            batch_size = 16  # Sicuro per singola T4
            workers = 4
            print(f"🔧 Kaggle singola GPU: T4 (15GB)")
        elif gpu_memory >= 24:  # GPU high-end (A100, V100)
            batch_size = 32
            workers = 8
        elif gpu_memory >= 16:  # GPU mid-range (T4, P100)
            batch_size = 24
            workers = 8
        else:  # GPU entry-level
            batch_size = 16
            workers = 4
elif torch.backends.mps.is_available():
    device = 'mps'
    batch_size = 16
    workers = 4
    print("🍎 Apple Silicon (MPS) rilevato")
else:
    device = 'cpu'
    batch_size = 8
    workers = 4
    print("⚠️ CPU rilevato - training sarà lento!")

# Libera memoria GPU se su Kaggle
if IS_KAGGLE and torch.cuda.is_available():
    torch.cuda.empty_cache()
    import gc
    gc.collect()
    print("🧹 Memoria GPU pulita")

TRAINING_PARAMS = {
    'data': str(updated_yaml_path),
    'epochs': 150,  # Aumenta a 200 per risultati migliori
    'imgsz': 640,
    'batch': batch_size,
    'lr0': 0.01,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3.0,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'box': 7.5,
    'cls': 0.5,
    'kobj': 1.0,
    'dfl': 1.5,
    'pose': 12.0,   # Importante per keypoints!
    'nbs': 64,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'degrees': 0.0,  # No rotation per preservare geometria
    'translate': 0.1,
    'scale': 0.5,
    'shear': 0.0,
    'perspective': 0.0,
    'flipud': 0.0,
    'fliplr': 0.5,
    'mosaic': 1.0,
    'mixup': 0.0,
    'copy_paste': 0.0,
    'patience': 50,
    'save': True,
    'save_period': -1,
    'cache': False,  # False per risparmiare RAM
    'device': device,
    'workers': workers,
    'project': 'runs/pose',
    'name': project_name,
    'exist_ok': False,
    'pretrained': True,
    'optimizer': 'auto',
    'verbose': True,
    'seed': 42,
    'deterministic': True,
    'single_cls': True,
    'rect': False,
    'cos_lr': False,
    'close_mosaic': 10,
    'resume': False,
    'amp': True,  # Automatic Mixed Precision per risparmiare memoria
    'fraction': 1.0,
    'profile': False,
    'freeze': None,
    'overlap_mask': True,
    'mask_ratio': 4,
    'dropout': 0.0,
    'val': True,
}

print("🚀 Configurazione Training:")
print(f"  - Modello: {MODEL_NAME}")
print(f"  - Checkpoint: COCO-pose (17 keypoint) → Transfer learning per 10 keypoint campo")
print(f"  - Dataset: {DATASET_ROOT.name}")
print(f"  - Epochs: {TRAINING_PARAMS['epochs']}")
print(f"  - Batch size: {TRAINING_PARAMS['batch']}", end="")
if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    print(f" (distribuito automaticamente su {torch.cuda.device_count()} GPU)")
else:
    print()
print(f"  - Image size: {TRAINING_PARAMS['imgsz']}")
print(f"  - Device: {TRAINING_PARAMS['device']}")
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    for i in range(gpu_count):
        print(f"  - GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB)")
    if gpu_count > 1:
        print(f"  ✅ Multi-GPU: YOLO rileva automaticamente {gpu_count} GPU (DataParallel)")
    # Mostra memoria disponibile
    print(f"  - GPU 0 Memory libera: {torch.cuda.mem_get_info(0)[0] / 1024**3:.2f} GB")
    if gpu_count > 1:
        print(f"  - GPU 1 Memory libera: {torch.cuda.mem_get_info(1)[0] / 1024**3:.2f} GB")
print(f"  - Workers: {TRAINING_PARAMS['workers']}")
print(f"  - AMP: {TRAINING_PARAMS['amp']} (Mixed Precision)")
print()

# Stima tempo con multi-GPU
if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    print("⏱️  Tempo stimato: 50-80 minuti (2 GPU automatico)")
    print("💡 YOLO userà DataParallel automaticamente - più stabile di DDP")
elif IS_KAGGLE:
    print("⏱️  Tempo stimato: 1-1.5 ore (Kaggle 1xT4)")
elif torch.cuda.is_available():
    print("⏱️  Tempo stimato: 1-2 ore (1 GPU)")
elif torch.backends.mps.is_available():
    print("⏱️  Tempo stimato: 2-3 ore (Apple Silicon MPS)")
else:
    print("⏱️  Tempo stimato: 8+ ore (CPU - non consigliato!)")
print()
print("✅ Pronto per il training! Esegui la cella successiva per iniziare.")

In [ ]:
# Avvia training
print("🚀 Inizio training...")
print("=" * 80)

results = model.train(**TRAINING_PARAMS)

print("=" * 80)
print("✅ Training completato!")
print()
print(f"📁 Risultati salvati in: runs/pose/{TRAINING_PARAMS['name']}")
print(f"📊 Metriche finali disponibili nei grafici seguenti")

## 6. Validazione e Metriche

Visualizziamo le metriche di training e validazione per valutare la performance del modello.

In [ ]:
# Path ai risultati
results_dir = Path(f"runs/pose/{TRAINING_PARAMS['name']}")

# Visualizza grafici di training
print("📊 Grafici di Training e Validazione:")
print()

# Results plot (loss curves, metrics, etc.)
results_img_path = results_dir / "results.png"
if results_img_path.exists():
    print("📈 Loss e Metriche durante il Training:")
    display(Image(filename=str(results_img_path)))
else:
    print(f"⚠️ Grafico risultati non trovato: {results_img_path}")

print()

# Confusion matrix
confusion_matrix_path = results_dir / "confusion_matrix.png"
if confusion_matrix_path.exists():
    print("🔢 Confusion Matrix:")
    display(Image(filename=str(confusion_matrix_path)))
else:
    print(f"⚠️ Confusion matrix non trovata: {confusion_matrix_path}")

print()

# Training batch examples
train_batch_path = results_dir / "train_batch0.jpg"
if train_batch_path.exists():
    print("🖼️ Esempi di Training Batch con Augmentation:")
    display(Image(filename=str(train_batch_path)))
else:
    print(f"⚠️ Training batch examples non trovati: {train_batch_path}")

In [ ]:
# Carica il modello migliore (best.pt) e valida sul validation set
best_model_path = results_dir / "weights" / "best.pt"

if best_model_path.exists():
    print(f"📥 Caricamento best model da: {best_model_path}")
    best_model = YOLO(str(best_model_path))
    
    print("🔍 Esecuzione validazione sul validation set...")
    val_results = best_model.val(data=str(updated_yaml_path), split='val', batch=16)
    
    print()
    print("=" * 80)
    print("📊 METRICHE FINALI - VALIDATION SET:")
    print("=" * 80)
    
    # Box metrics
    print("\n📦 Bounding Box Detection:")
    print(f"  - mAP50: {val_results.box.map50:.4f}")
    print(f"  - mAP50-95: {val_results.box.map:.4f}")
    print(f"  - Precision: {val_results.box.mp:.4f}")
    print(f"  - Recall: {val_results.box.mr:.4f}")
    
    # Keypoint metrics
    if hasattr(val_results, 'pose'):
        print("\n🎯 Keypoint Detection:")
        print(f"  - mAP50 (keypoints): {val_results.pose.map50:.4f}")
        print(f"  - mAP50-95 (keypoints): {val_results.pose.map:.4f}")
        print(f"  - Precision (keypoints): {val_results.pose.mp:.4f}")
        print(f"  - Recall (keypoints): {val_results.pose.mr:.4f}")
    
    # Per-class metrics
    print("\n📋 Metriche per Classe:")
    for i, class_name in enumerate(data_config['names']):
        if i < len(val_results.box.ap):
            ap50 = val_results.box.ap50[i]
            print(f"  - {class_name}: AP50 = {ap50:.4f}")
    
    print("\n" + "=" * 80)
    
else:
    print(f"❌ Modello best.pt non trovato in: {best_model_path}")
    print("Assicurati che il training sia completato correttamente.")

In [ ]:
# Visualizza esempi di validazione con predizioni
val_batch_paths = sorted(results_dir.glob("val_batch*_pred.jpg"))

if val_batch_paths:
    print("🖼️ Esempi di Predizioni sul Validation Set:")
    print("(Ground Truth in verde, Predizioni in rosso)")
    print()
    
    for i, val_batch_path in enumerate(val_batch_paths[:3]):  # Mostra primi 3 batch
        print(f"Validation Batch {i}:")
        display(Image(filename=str(val_batch_path)))
        print()
else:
    print("⚠️ Nessun batch di validazione trovato.")
    print("I batch di validazione potrebbero essere disabilitati nelle impostazioni.")

## 7. Test su Immagini di Esempio

Testiamo il modello addestrato su immagini del test set per vedere la qualità delle predizioni.

In [ ]:
# Test su immagini del test set
test_images = sorted((DATASET_ROOT / "test" / "images").glob("*.jpg"))[:6]

if not test_images:
    print("⚠️ Nessuna immagine trovata nel test set!")
else:
    print(f"🧪 Testing su {len(test_images)} immagini del test set...")
    print()
    
    # Inferenza
    results_list = best_model.predict(
        source=[str(img) for img in test_images],
        save=False,
        conf=0.25,  # Confidence threshold
        iou=0.45,   # NMS IoU threshold
        verbose=False
    )
    
    # Visualizza risultati
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for i, (result, img_path) in enumerate(zip(results_list, test_images)):
        # Ottieni immagine con annotazioni
        annotated_img = result.plot()
        annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
        
        axes[i].imshow(annotated_img)
        axes[i].set_title(f"Test Image {i+1}: {img_path.name}", fontsize=10)
        axes[i].axis('off')
        
        # Stampa dettagli predizioni
        n_detections = len(result.boxes)
        print(f"Image {i+1} ({img_path.name}): {n_detections} detections")
        
        if n_detections > 0:
            for j, box in enumerate(result.boxes):
                class_id = int(box.cls[0])
                conf = float(box.conf[0])
                class_name = data_config['names'][class_id]
                print(f"  - Detection {j+1}: {class_name} (conf: {conf:.3f})")
        print()
    
    plt.tight_layout()
    plt.show()
    
    print("✅ Test completato!")

In [ ]:
# Confronto Ground Truth vs Predizioni
def compare_gt_pred(image_path, label_path, model, class_names):
    """Confronta ground truth e predizioni fianco a fianco"""
    # Leggi immagine
    img = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    
    # Ground Truth
    gt_img = img_rgb.copy()
    annotations = parse_yolo_keypoints(label_path)
    colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0)]
    
    for ann in annotations:
        class_id = ann['class_id']
        cx, cy, bw, bh = ann['bbox']
        
        cx_px, cy_px = int(cx * w), int(cy * h)
        bw_px, bh_px = int(bw * w), int(bh * h)
        
        x1 = int(cx_px - bw_px/2)
        y1 = int(cy_px - bh_px/2)
        x2 = int(cx_px + bw_px/2)
        y2 = int(cy_px + bh_px/2)
        
        color = colors[class_id % len(colors)]
        cv2.rectangle(gt_img, (x1, y1), (x2, y2), color, 2)
        
        label = f"GT: {class_names[class_id]}"
        cv2.putText(gt_img, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 
                   0.5, color, 2)
        
        for kpt_x, kpt_y, visibility in ann['keypoints']:
            if visibility > 0:
                kpt_x_px = int(kpt_x * w)
                kpt_y_px = int(kpt_y * h)
                cv2.circle(gt_img, (kpt_x_px, kpt_y_px), 5, color, -1)
                cv2.circle(gt_img, (kpt_x_px, kpt_y_px), 7, (255, 255, 255), 2)
    
    # Predizioni
    results = model.predict(source=str(image_path), save=False, conf=0.25, verbose=False)
    pred_img = results[0].plot()
    pred_img = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB)
    
    return gt_img, pred_img

# Visualizza confronti
print("📊 Confronto Ground Truth (sinistra) vs Predizioni (destra):")
print()

test_images_compare = sorted((DATASET_ROOT / "test" / "images").glob("*.jpg"))[:3]

for idx, img_path in enumerate(test_images_compare):
    label_path = DATASET_ROOT / "test" / "labels" / f"{img_path.stem}.txt"
    
    if label_path.exists():
        gt_img, pred_img = compare_gt_pred(img_path, label_path, best_model, data_config['names'])
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
        
        ax1.imshow(gt_img)
        ax1.set_title(f"Ground Truth - {img_path.name}", fontsize=12, fontweight='bold')
        ax1.axis('off')
        
        ax2.imshow(pred_img)
        ax2.set_title(f"Predizioni - {img_path.name}", fontsize=12, fontweight='bold')
        ax2.axis('off')
        
        plt.tight_layout()
        plt.show()
        print()

print("✅ Confronto completato!")

## 8. Salvataggio Modello e Export

Salviamo copie dei modelli nella directory `../models/field_keypoints/` ed esportiamo in diversi formati.

In [ ]:
# Copia modelli nella directory del progetto
if IS_KAGGLE:
    # Su Kaggle salva in /kaggle/working/ (scaricabile come output)
    output_dir = Path("/kaggle/working/field_skeleton")
elif IS_COLAB:
    # Su Colab salva in Google Drive
    output_dir = Path("/content/drive/MyDrive/Padel_Models/field_skeleton")
else:
    # Locale
    output_dir = Path("../models/field_skeleton")

output_dir.mkdir(parents=True, exist_ok=True)

print(f"📁 Directory output: {output_dir.absolute()}")
print()

# Copia best.pt
if best_model_path.exists():
    dest_best = output_dir / "best.pt"
    shutil.copy(best_model_path, dest_best)
    print(f"✅ Best model salvato in: {dest_best}")
    print(f"   Dimensione: {dest_best.stat().st_size / 1024**2:.2f} MB")

# Copia last.pt
last_model_path = results_dir / "weights" / "last.pt"
if last_model_path.exists():
    dest_last = output_dir / "last.pt"
    shutil.copy(last_model_path, dest_last)
    print(f"✅ Last model salvato in: {dest_last}")
    print(f"   Dimensione: {dest_last.stat().st_size / 1024**2:.2f} MB")

# Copia tutti i risultati di training
results_backup = output_dir / "training_results"
if results_dir.exists():
    if results_backup.exists():
        shutil.rmtree(results_backup)
    shutil.copytree(results_dir, results_backup)
    print(f"✅ Risultati training salvati in: {results_backup}")

print()
print(f"💾 Tutti i file salvati in: {output_dir.absolute()}!")

if IS_KAGGLE:
    print()
    print("📥 Su Kaggle: I file in /kaggle/working/ sono scaricabili come output del notebook")
    print("   Dopo il training, vai su 'Data' → 'Output' per scaricare i modelli")
elif IS_COLAB:
    print()
    print("☁️  Su Colab: I file sono salvati su Google Drive e persistono tra sessioni")

print(f"📊 Risultati training originali in: {results_dir.absolute()}")

In [ ]:
# Export modello in formati ottimizzati
print("🔄 Export modello in diversi formati...")
print()

# ONNX (cross-platform inference)
try:
    onnx_path = best_model.export(format='onnx', simplify=True)
    onnx_dest = output_dir / "best.onnx"
    shutil.copy(onnx_path, onnx_dest)
    print(f"✅ ONNX export: {onnx_dest}")
    print(f"   Dimensione: {onnx_dest.stat().st_size / 1024**2:.2f} MB")
except Exception as e:
    print(f"❌ ONNX export fallito: {e}")

print()

# TorchScript (PyTorch optimized)
try:
    torchscript_path = best_model.export(format='torchscript')
    ts_dest = output_dir / "best.torchscript"
    shutil.copy(torchscript_path, ts_dest)
    print(f"✅ TorchScript export: {ts_dest}")
    print(f"   Dimensione: {ts_dest.stat().st_size / 1024**2:.2f} MB")
except Exception as e:
    print(f"❌ TorchScript export fallito: {e}")

print()

# TensorRT (NVIDIA GPU optimized - richiede GPU NVIDIA)
if torch.cuda.is_available():
    try:
        trt_path = best_model.export(format='engine', half=True)  # FP16 for speed
        trt_dest = output_dir / "best.engine"
        shutil.copy(trt_path, trt_dest)
        print(f"✅ TensorRT export: {trt_dest}")
        print(f"   Dimensione: {trt_dest.stat().st_size / 1024**2:.2f} MB")
    except Exception as e:
        print(f"⚠️ TensorRT export fallito: {e}")
        print("   (Normale se non hai una GPU NVIDIA compatibile)")
else:
    print("⚠️ TensorRT export saltato (richiede GPU NVIDIA)")

print()
print("✅ Export completati!")

## 🎉 Training Completato!

### Riepilogo

Il modello **YOLOv11-pose** è stato addestrato con successo per il riconoscimento del **field skeleton** (10 keypoint geometrici) tramite **transfer learning da checkpoint COCO-pose**.

### File Salvati

Tutti i seguenti file sono stati salvati in `../models/field_skeleton/`:

- **best.pt**: Modello con le migliori performance (usalo per l'inferenza)
- **last.pt**: Ultimo checkpoint del training
- **best.onnx**: Export ONNX per deployment cross-platform
- **best.torchscript**: Export TorchScript ottimizzato per PyTorch
- **best.engine** (opzionale): Export TensorRT per GPU NVIDIA
- **training_results/**: Copia completa di tutti i risultati (grafici, metriche, ecc.)

**Path originali training**: I risultati completi sono anche in `runs/pose/field_skeleton_[model]/`

### Schema Keypoint (10 punti)

```
0: corner_back_left       - Angolo posteriore sinistro
1: corner_back_right      - Angolo posteriore destro  
2: corner_front_left      - Angolo anteriore sinistro
3: corner_front_right     - Angolo anteriore destro
4: net_top_left          - Palo rete sinistro lato superiore
5: net_top_right         - Palo rete destro lato superiore
6: net_bottom_left       - Palo rete sinistro lato inferiore
7: net_bottom_right      - Palo rete destro lato inferiore
8: service_line_top      - Centro linea servizio campo superiore
9: service_line_bottom   - Centro linea servizio campo inferiore
```

### Come Usare il Modello

```python
from ultralytics import YOLO
import cv2

# Carica il modello addestrato
model = YOLO('../models/field_skeleton/best.pt')

# Inferenza su nuove immagini
results = model.predict('path/to/image.jpg', conf=0.25)

# Accedi ai keypoint del campo
for result in results:
    if len(result.keypoints) > 0:
        kpts = result.keypoints.xy[0]  # Shape: [10, 2] (x, y)
        visibility = result.keypoints.conf[0]  # Shape: [10] (confidence)
        
        # Estrai coordinate specifiche
        corner_back_left = kpts[0]
        net_top_left = kpts[4]
        # ... etc
        
        # Ricostruisci geometria campo
        corners = kpts[0:4]  # 4 angoli
        net_points = kpts[4:8]  # 4 punti rete
```

### Applicazioni

1. **Perimetro campo**: Estrai `corners` (kpts 0-3) → ricostruisci perimetro con interpolazione
2. **Divisione squadre**: Usa `net_points` (kpts 4-7) per linea di separazione centrale
3. **Homography 2D**: Trasforma keypoint 3D→2D per vista dall'alto del campo
4. **Tracking assistito**: Usa geometria campo come contesto per player/ball tracking

### Prossimi Passi

1. **Test su video personali**: Valida il modello sui video in `data/personal/`
2. **Integrazione FieldDetector**: Aggiorna `detection/field_detector.py` per usare il nuovo modello
3. **Geometria campo**: Implementa funzioni per estrarre perimetro, rete, e coordinate normalizzate
4. **Homography**: Implementa trasformazione prospettica usando i 4 corner + 2 punti rete

### Vantaggi vs Approccio Precedente

- ✅ **Skeleton strutturato** vs keypoint sparsi
- ✅ **10 punti geometrici fissi** vs N×1 keypoint casuali
- ✅ **Homography diretta** con corrispondenze note
- ✅ **1 detection robusta** vs 30+ detection da unire
- ✅ **Gestione occlusioni** migliore con skeleton coerente

### Metriche Finali

Controlla le metriche nella sezione 6 per valutare la performance del modello.

In [ ]:
# Test rapido di inferenza su una singola immagine
print("🧪 Test di Inferenza Rapida")
print("=" * 80)
print()

# Prendi una immagine random dal test set
test_img = next((DATASET_ROOT / "test" / "images").glob("*.jpg"))

# Inferenza
result = best_model.predict(str(test_img), conf=0.25, save=False, verbose=False)[0]

# Visualizza
annotated = result.plot()
annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(annotated)
plt.title(f"Inferenza su: {test_img.name}", fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

# Stampa dettagli
n_detections = len(result.boxes)
print(f"\n📊 Risultati:")
print(f"  - Immagine: {test_img.name}")
print(f"  - Detections: {n_detections}")
print()

if n_detections > 0:
    print("Dettagli detections:")
    for i, box in enumerate(result.boxes):
        class_id = int(box.cls[0])
        conf = float(box.conf[0])
        class_name = data_config['names'][class_id]
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        
        print(f"  {i+1}. {class_name}")
        print(f"     - Confidence: {conf:.3f}")
        print(f"     - Bbox: [{int(x1)}, {int(y1)}, {int(x2)}, {int(y2)}]")
        
        # Keypoints se disponibili
        if hasattr(result, 'keypoints') and result.keypoints is not None:
            kpts = result.keypoints.data[i].cpu().numpy()
            print(f"     - Keypoints: {len(kpts)} punti rilevati")

print()
print("=" * 80)
print("✅ Test completato! Il modello è pronto per l'uso.")